In [ ]:
!pip install faker

#  Smart Electricity Consumption Prediction & Energy Optimization System

## Step 1: Synthetic Dataset Generation
*(Generating a realistic dataset of 1,500 records using the Faker library, based on real-world energy consumption formulas and weather patterns.)*

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random

print("Initializing Faker for Smart Electricity Dataset Generation...")

# Setup Faker and Seeds for reproducibility
fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

num_records = 1500 # Generating 1500 records for a robust ML model
data = []

house_types = ['Apartment', 'Single Family Home', 'Villa', 'Portion']

for _ in range(num_records):
    # Using Faker for realistic IDs and Dates
    household_id = fake.uuid4()[:8].upper()
    simulated_date = fake.date_between(start_date='-1y', end_date='today')
    day_of_week = simulated_date.strftime('%A')

    # Season logic based on the generated month
    month = simulated_date.month
    if month in [5, 6, 7, 8, 9]:
        season = 'Summer'
    elif month in [11, 12, 1, 2]:
        season = 'Winter'
    else:
        season = 'Spring/Autumn'

    # Demographics & House details
    house = fake.random_element(elements=house_types)
    family_members = fake.random_int(min=1, max=8)

    if house == 'Villa':
        rooms = fake.random_int(min=5, max=10)
    elif house == 'Apartment':
        rooms = fake.random_int(min=2, max=4)
    else:
        rooms = fake.random_int(min=3, max=6)

    is_holiday = 1 if day_of_week in ['Saturday', 'Sunday'] else fake.random_elements(elements=(0, 1), length=1, unique=False)[0]

    # Weather & Appliance Usage Logic
    if season == 'Summer':
        temp = round(random.uniform(35.0, 45.0), 1)
        ac_hours = random.uniform(6, 16) if temp > 38 else random.uniform(2, 8)
        fan_hours = random.uniform(12, 24)
        water_motor_hours = random.uniform(1.5, 3.0)
    elif season == 'Winter':
        temp = round(random.uniform(5.0, 20.0), 1)
        ac_hours = 0.0
        fan_hours = random.uniform(0, 4)
        water_motor_hours = random.uniform(0.5, 1.5)
    else:
        temp = round(random.uniform(20.0, 32.0), 1)
        ac_hours = random.uniform(0, 4)
        fan_hours = random.uniform(4, 12)
        water_motor_hours = random.uniform(1.0, 2.0)

    fridge_hours = random.uniform(10, 14)
    washing_machine_hours = random.uniform(1, 3) if is_holiday else random.uniform(0, 1.0)
    lighting_hours = random.uniform(4, 8) if season == 'Summer' else random.uniform(6, 12)

    # Target Variable Calculation: Daily Electricity Consumption (kWh)
    # AC(1.5kW), Fan(0.075kW), Fridge(0.15kW), Washing Machine(0.5kW), Motor(1.5kW), Lights(0.1kW per room)
    consumption = (ac_hours * 1.5) + (fan_hours * 0.075 * rooms) + (fridge_hours * 0.15) + \
                  (washing_machine_hours * 0.5) + (water_motor_hours * 1.5) + (lighting_hours * 0.1 * rooms)

    # Add realistic noise
    consumption += random.uniform(-1.5, 2.5)
    if consumption < 1:
        consumption = 1.0

    # Append to data list
    data.append([
        household_id, simulated_date, house, family_members, rooms, round(ac_hours, 1), round(fan_hours, 1),
        round(fridge_hours, 1), round(washing_machine_hours, 1), round(water_motor_hours, 1),
        round(lighting_hours, 1), temp, day_of_week, season, is_holiday, round(consumption, 2)
    ])

# Define Columns (15 Features + Target)
columns = [
    'Household ID', 'Date', 'House Type', 'Family Members', 'Number of Rooms', 'AC Usage (Hours)',
    'Fan Usage (Hours)', 'Refrigerator Usage (Hours)', 'Washing Machine Usage (Hours)',
    'Water Motor Usage (Hours)', 'Lighting (Hours)', 'Outdoor Temperature (C)',
    'Day of Week', 'Season', 'Is Holiday', 'Daily Consumption (kWh)'
]

df = pd.DataFrame(data, columns=columns)

# Save to CSV
df.to_csv('faker_electricity_consumption_dataset.csv', index=False)

print("Dataset Generated Successfully")
print(f"Dataset Shape: {df.shape}")
display(df.head())

---
## Step 2: Data Preprocessing & Exploratory Data Analysis (EDA)
*(Cleaning data, handling categorical variables via One-Hot Encoding, scaling features, and visualizing correlation patterns.)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

print("Data Preprocessing & EDA...\n")

# Load Dataset
df = pd.read_csv('faker_electricity_consumption_dataset.csv')

# Data Cleaning & Preprocessing
# Duplicate Removal
df.drop_duplicates(inplace=True)

# Missing Value Handling
df.fillna(df.median(numeric_only=True), inplace=True)

# Drop columns not needed for ML mathematical modeling
df_ml = df.drop(['Household ID', 'Date'], axis=1)

# Outlier Detection using Boxplot for Target Variable
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_ml['Daily Consumption (kWh)'], color='orange')
plt.title('Outlier Detection: Daily Consumption (kWh)')
plt.show()

# Exploratory Data Analysis (EDA)
# Statistical Summary
print("\n--- Statistical Summary ---")
display(df_ml.describe())

# Feature Distribution (Histogram)
plt.figure(figsize=(8, 4))
sns.histplot(df_ml['Daily Consumption (kWh)'], bins=30, kde=True, color='blue')
plt.title('Distribution of Daily Electricity Consumption')
plt.xlabel('kWh')
plt.ylabel('Frequency')
plt.show()

# Scatter Plot: AC Usage vs Consumption (Feature Relationship)
plt.figure(figsize=(8, 4))
sns.scatterplot(x='AC Usage (Hours)', y='Daily Consumption (kWh)', hue='Season', data=df_ml)
plt.title('Feature Relationship: AC Usage vs Consumption')
plt.show()

# Feature Encoding
# One-Hot Encoding for categorical features
df_ml = pd.get_dummies(df_ml, columns=['House Type', 'Day of Week', 'Season'], drop_first=True)

# Feature Scaling / Normalization
# Scaling numeric columns for better regression performance
num_cols = ['Family Members', 'Number of Rooms', 'AC Usage (Hours)', 'Fan Usage (Hours)',
            'Refrigerator Usage (Hours)', 'Washing Machine Usage (Hours)', 'Water Motor Usage (Hours)',
            'Lighting (Hours)', 'Outdoor Temperature (C)']

scaler = StandardScaler()
df_ml[num_cols] = scaler.fit_transform(df_ml[num_cols])

# Correlation Analysis (Heatmap)
plt.figure(figsize=(16, 10))
correlation_matrix = df_ml.corr()
# Turned off annotations (annot=False) because there are many features after encoding
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap: All Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Display correlations with target variable to find business insights
print("\n--- Feature Correlation with Target (Daily Consumption) ---")
print(correlation_matrix['Daily Consumption (kWh)'].sort_values(ascending=False).head(10))

# Save the preprocessed dataset
df_ml.to_csv('preprocessed_electricity_data.csv', index=False)

print("\n Data is cleaned, scaled, encoded, and ready for Model Training.")

---
## Step 3: Machine Learning Model Development & Evaluation
*(Training multiple regression models including Linear Regression, Random Forest, and XGBoost, and selecting the best performer based on R² Score.)*

In [ ]:
# Model Training & Evaluation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print("Model Training & Evaluation...\n")

# Load Preprocessed Data
df_ml = pd.read_csv('preprocessed_electricity_data.csv')

# Separate Features (X) and Target (y)
X = df_ml.drop('Daily Consumption (kWh)', axis=1)
y = df_ml['Daily Consumption (kWh)']

# Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize 3 Different Regression Models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []
trained_models = {}

# Train and Evaluate each model
for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    trained_models[name] = model

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate Regression Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

# Display Model Comparison Report
results_df = pd.DataFrame(results)
print("--- Model Performance Comparison ---")
display(results_df)

# Visual Comparison (R2 Score)
plt.figure(figsize=(8, 5))
bars = plt.bar(results_df['Model'], results_df['R2 Score'], color=['#3498db', '#2ecc71', '#e74c3c'])
plt.title('Model Comparison: R² Score (Higher is Better)', fontsize=14)
plt.ylabel('R² Score')
plt.ylim(0, 1.1)

# Add values on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, round(yval, 3), ha='center', va='bottom', fontweight='bold')

plt.show()

# Save the Best Performing Model
best_model_name = results_df.loc[results_df['R2 Score'].idxmax()]['Model']
best_model = trained_models[best_model_name]

print(f"\n Best Model Selected: {best_model_name}")

# Save model and feature names for deployment
joblib.dump(best_model, 'best_electricity_model.pkl')
joblib.dump(list(X.columns), 'model_features.pkl')
# Save the scaler as well for real-time input scaling
joblib.dump(scaler, 'scaler.pkl')

print("Best model, features, and scaler saved successfully.")

---
## Step 4: Live Energy Optimization Web App (Gradio)
*(Deploying an interactive prediction interface that calculates estimated bills, efficiency scores, and provides smart energy-saving recommendations.)*

In [ ]:
# Gradio Web App & Smart Recommendations
import gradio as gr
import pandas as pd
import numpy as np
import joblib

print("Deploying Smart Energy Prediction System...")

# Load Saved ML Artifacts
model = joblib.load('best_electricity_model.pkl')
model_features = joblib.load('model_features.pkl')
scaler = joblib.load('scaler.pkl')

# Feature list for scaling
num_cols = ['Family Members', 'Number of Rooms', 'AC Usage (Hours)', 'Fan Usage (Hours)',
            'Refrigerator Usage (Hours)', 'Washing Machine Usage (Hours)', 'Water Motor Usage (Hours)',
            'Lighting (Hours)', 'Outdoor Temperature (C)']

def predict_energy(house_type, family_members, rooms, ac, fan, fridge, washing, motor, lighting, temp, day, season, is_holiday):
    # Prepare Input DataFrame
    input_data = pd.DataFrame([[
        house_type, family_members, rooms, ac, fan, fridge, washing, motor, lighting, temp, day, season, int(is_holiday)
    ]], columns=[
        'House Type', 'Family Members', 'Number of Rooms', 'AC Usage (Hours)', 'Fan Usage (Hours)',
        'Refrigerator Usage (Hours)', 'Washing Machine Usage (Hours)', 'Water Motor Usage (Hours)',
        'Lighting (Hours)', 'Outdoor Temperature (C)', 'Day of Week', 'Season', 'Is Holiday'
    ])

    # Preprocessing & Encoding
    input_ml = pd.get_dummies(input_data, columns=['House Type', 'Day of Week', 'Season'])
    input_ml[num_cols] = scaler.transform(input_ml[num_cols])
    input_ml = input_ml.reindex(columns=model_features, fill_value=0)

    # Model Prediction
    daily_kwh = float(model.predict(input_ml)[0])
    monthly_kwh = daily_kwh * 30

    # Smart Calculations (Assuming 45 PKR per kWh avg rate)
    monthly_bill = monthly_kwh * 45

    # Energy Efficiency Score Calculation
    per_person_usage = daily_kwh / family_members
    if per_person_usage < 3: score, grade = 95, "A+ (Excellent)"
    elif per_person_usage < 5: score, grade = 80, "B (Good)"
    elif per_person_usage < 8: score, grade = 60, "C (Average)"
    else: score, grade = 40, "D (Needs Improvement)"

    # Peak Hours & Recommendations Engine
    peak_hours = "6:00 PM to 10:00 PM" if ac > 4 or lighting > 6 else "12:00 PM to 4:00 PM"

    recommendations = "💡 **Personalized Energy Optimization Insights:**\n"
    if ac > 6:
        recommendations += "- **AC Optimization:** You are using AC for extended hours. Try setting it to 26°C and using a fan simultaneously to save up to 20% cooling costs.\n"
    if washing > 1.5 or motor > 1.5:
        recommendations += f"- **Load Shifting:** Run heavy appliances (Washing Machine & Motor) outside the peak hours of {peak_hours} to reduce grid strain.\n"
    if lighting > 8:
        recommendations += "- **Lighting:** High lighting hours detected. Switch to energy-efficient LED bulbs to significantly drop daily load.\n"
    if grade == "D (Needs Improvement)":
        recommendations += "- **Alert:** Your per-capita energy usage is very high compared to similar households. Consider an immediate energy audit.\n"
    else:
        recommendations += "- **Great Job:** Your baseline consumption is well-managed!\n"

    # Formatting Outputs
    res_daily = f"{daily_kwh:.2f} kWh"
    res_monthly = f"{monthly_kwh:.2f} kWh"
    res_bill = f"Rs. {monthly_bill:,.2f}"
    res_score = f"{score}/100 - {grade}"

    return res_daily, res_monthly, res_bill, peak_hours, res_score, recommendations

# ----------------- Gradio UI Layout -----------------
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# ⚡ Smart Electricity Consumption Prediction & Energy Optimization System")
    gr.Markdown("Enter household details and appliance usage to predict energy consumption and receive smart recommendations.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🏠 Household & Environment")
            house_type = gr.Dropdown(['Apartment', 'Single Family Home', 'Villa', 'Portion'], label="House Type", value='Single Family Home')
            season = gr.Dropdown(['Summer', 'Winter', 'Spring/Autumn'], label="Season", value='Summer')
            day = gr.Dropdown(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], label="Day of Week", value='Monday')
            temp = gr.Slider(5.0, 50.0, step=0.5, label="Outdoor Temperature (°C)", value=35.0)
            family = gr.Slider(1, 10, step=1, label="Family Members", value=4)
            rooms = gr.Slider(1, 15, step=1, label="Number of Rooms", value=4)
            is_holiday = gr.Checkbox(label="Is it a Holiday?", value=False)

        with gr.Column():
            gr.Markdown("### 🔌 Appliance Daily Usage (Hours)")
            ac = gr.Slider(0, 24, step=0.5, label="Air Conditioner", value=4.0)
            fan = gr.Slider(0, 24, step=0.5, label="Fans", value=12.0)
            fridge = gr.Slider(0, 24, step=0.5, label="Refrigerator (Compressor Active)", value=12.0)
            washing = gr.Slider(0, 5, step=0.5, label="Washing Machine", value=0.5)
            motor = gr.Slider(0, 5, step=0.5, label="Water Motor", value=1.0)
            lighting = gr.Slider(0, 24, step=0.5, label="Lighting", value=6.0)

            predict_btn = gr.Button("Predict & Optimize 🚀", variant="primary")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📊 Prediction Results")
            out_daily = gr.Textbox(label="Predicted Daily Consumption")
            out_monthly = gr.Textbox(label="Estimated Monthly Usage")
            out_bill = gr.Textbox(label="Estimated Monthly Bill (PKR)")
        with gr.Column():
            gr.Markdown("### 🌿 Energy Intelligence")
            out_peak = gr.Textbox(label="Peak Usage Hours")
            out_score = gr.Textbox(label="Energy Efficiency Score")
            out_recs = gr.Markdown()

    predict_btn.click(
        fn=predict_energy,
        inputs=[house_type, family, rooms, ac, fan, fridge, washing, motor, lighting, temp, day, season, is_holiday],
        outputs=[out_daily, out_monthly, out_bill, out_peak, out_score, out_recs]
    )

app.launch(share=True)